# LLM-Based Task Planning for Robotics

This notebook demonstrates how to use Large Language Models (LLMs) to convert natural language commands into structured robot task plans.

## Learning Objectives

By the end of this notebook, you will:
1. Understand prompt engineering for robotic task planning
2. Learn JSON schema validation for structured outputs
3. Implement retry logic and error handling for LLM APIs
4. Design safety constraints for robot planning
5. Evaluate plan feasibility and confidence scoring

## Prerequisites

- Completed `01_voice_to_text.ipynb`
- OpenAI API key (set in `.env` file)
- Understanding of JSON data structures

## Setup

In [ ]:
# Import required libraries
import sys
sys.path.append('../scripts')

import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
from llm_planner import LLMPlanner, Task, TaskPlan, FeasibilityAssessment
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Libraries imported")
print(f"✓ OpenAI API Key: {'Set' if os.getenv('OPENAI_API_KEY') else 'NOT SET'}")

## Part 1: Understanding Task Planning

### What is Task Planning?

Task planning converts high-level goals into sequences of executable actions:

```
Natural Language → LLM → Structured Plan → Robot Actions
"Get the cup"        ↓         ↓              ↓
                 [navigate]  [detect]     [grasp]
```

### Task Types in VLA System

| Type | Purpose | Parameters | Example |
|------|---------|------------|--------|
| `navigate` | Move to location | target, position [x,y,z] | Go to kitchen |
| `detect` | Find objects | object_name, confidence | Locate red cup |
| `manipulate` | Pick/place | object, action (pick/place), target | Grasp cup |
| `inspect` | Visual check | target, check_type | Verify grasped object |

## Part 2: JSON Schema for Structured Output

LLMs can be unreliable without constraints. JSON schemas enforce structure.

In [ ]:
# Example task plan schema
example_schema = {
    "type": "object",
    "required": ["tasks", "feasibility"],
    "properties": {
        "tasks": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["task_id", "task_type", "parameters"],
                "properties": {
                    "task_id": {"type": "integer"},
                    "task_type": {"type": "string", "enum": ["navigate", "detect", "manipulate", "inspect"]},
                    "parameters": {"type": "object"},
                    "preconditions": {"type": "array"},
                    "expected_duration": {"type": "number"}
                }
            }
        },
        "feasibility": {
            "type": "object",
            "properties": {
                "is_feasible": {"type": "boolean"},
                "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0}
            }
        }
    }
}

print("Task Plan JSON Schema:")
print(json.dumps(example_schema, indent=2))

## Part 3: Prompt Engineering for Task Planning

### Key Principles:
1. **Clear Role Definition**: "You are a robotic task planner"
2. **Explicit Constraints**: Safety limits, workspace bounds
3. **Structured Output**: Force JSON format
4. **Few-Shot Examples**: Show desired outputs
5. **Error Handling**: Specify fallback behaviors

In [ ]:
# Example system prompt structure
system_prompt_template = """You are a robotic task planner.

AVAILABLE TASK TYPES:
- navigate: Move to location
- detect: Find objects
- manipulate: Pick/place
- inspect: Visual check

SAFETY CONSTRAINTS:
- Navigate before manipulation
- Detect before grasping
- Workspace: x[-5,5], y[-5,5], z[0,3] meters

Return ONLY valid JSON."""

print("System Prompt:")
print(system_prompt_template)
print("\n" + "="*60)
print("Key Elements:")
print("1. Role definition (task planner)")
print("2. Available actions (4 task types)")
print("3. Safety constraints (ordered dependencies)")
print("4. Output format (JSON only)")

## Part 4: Initialize LLM Planner

**Note**: This requires a valid OpenAI API key. If not set, some cells will fail.

In [ ]:
# Initialize planner
try:
    planner = LLMPlanner(
        model="gpt-4",  # or "gpt-3.5-turbo" for faster/cheaper
        temperature=0.0  # Deterministic output
    )
    print("✓ LLM Planner initialized")
    print(f"  Model: {planner.model}")
    print(f"  Temperature: {planner.temperature}")
except ValueError as e:
    print(f"❌ Error: {e}")
    print("\nTo use this notebook:")
    print("1. Get OpenAI API key: https://platform.openai.com/api-keys")
    print("2. Create .env file in code-examples/ directory")
    print("3. Add: OPENAI_API_KEY=sk-...")
    planner = None

## Part 5: Simple Task Planning Example

In [ ]:
if planner:
    # Simple navigation command
    command = "Navigate to the kitchen"
    
    print(f"Command: '{command}'")
    print("\nGenerating plan...\n")
    
    plan = planner.generate_plan(command)
    
    # Display plan
    print("\n" + "="*60)
    print("Generated Plan:")
    print("="*60)
    print(planner.plan_to_json(plan))
    print("="*60)
    
    # Analyze plan
    print(f"\nNumber of tasks: {len(plan.tasks)}")
    print(f"Feasible: {plan.feasibility.is_feasible}")
    print(f"Confidence: {plan.feasibility.confidence:.2%}")
    print(f"Total duration: {sum(t.expected_duration for t in plan.tasks):.1f}s")
else:
    print("⚠️ Planner not initialized (API key required)")

## Part 6: Complex Multi-Step Planning

In [ ]:
if planner:
    # Complex command requiring multiple steps
    complex_command = "Navigate to the kitchen and pick up the red cup from the table"
    
    # Add environment context
    context = {
        "known_objects": ["red cup", "blue plate", "table", "chair"],
        "known_locations": ["kitchen", "living room", "bedroom"],
        "robot_position": [0.0, 0.0, 0.0]
    }
    
    print(f"Command: '{complex_command}'")
    print(f"\nContext: {json.dumps(context, indent=2)}")
    print("\nGenerating plan...\n")
    
    complex_plan = planner.generate_plan(complex_command, environment_context=context)
    
    print("\n" + "="*60)
    print("Generated Plan:")
    print("="*60)
    print(planner.plan_to_json(complex_plan))
    print("="*60)
    
    # Analyze task dependencies
    print("\nTask Sequence:")
    for task in complex_plan.tasks:
        precond_str = f"after {task.preconditions}" if task.preconditions else "(no dependencies)"
        print(f"  {task.task_id}. {task.task_type}: {task.parameters.get('target', 'N/A')} {precond_str}")
else:
    print("⚠️ Planner not initialized (API key required)")

## Part 7: Visualizing Task Dependencies

Plans often have dependencies: Task B cannot start until Task A completes.

In [ ]:
def visualize_task_plan(plan: TaskPlan, title="Task Plan Dependencies"):
    """
    Visualize task plan as a directed acyclic graph (DAG).
    """
    G = nx.DiGraph()
    
    # Add nodes
    for task in plan.tasks:
        label = f"T{task.task_id}\n{task.task_type}\n{task.expected_duration}s"
        G.add_node(task.task_id, label=label, task_type=task.task_type)
    
    # Add edges (dependencies)
    for task in plan.tasks:
        for precond_id in task.preconditions:
            G.add_edge(precond_id, task.task_id)
    
    # Layout
    pos = nx.spring_layout(G, k=2, iterations=50)
    
    # Color by task type
    color_map = {
        'navigate': '#3498db',
        'detect': '#2ecc71',
        'manipulate': '#e74c3c',
        'inspect': '#f39c12'
    }
    
    node_colors = [color_map.get(G.nodes[n]['task_type'], '#95a5a6') for n in G.nodes()]
    labels = nx.get_node_attributes(G, 'label')
    
    # Plot
    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, 
            node_color=node_colors,
            node_size=3000,
            labels=labels,
            font_size=8,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            edge_color='#7f8c8d',
            width=2,
            alpha=0.9)
    
    plt.title(title, fontsize=14, fontweight='bold')
    
    # Legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=task_type.capitalize())
        for task_type, color in color_map.items()
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Visualize the complex plan
if planner and 'complex_plan' in locals():
    visualize_task_plan(complex_plan, "Kitchen Cup Retrieval Plan")
else:
    print("⚠️ No plan to visualize (run previous cells first)")

## Part 8: Feasibility Assessment

Not all commands can be executed. LLMs should assess feasibility.

In [ ]:
if planner:
    # Test infeasible commands
    infeasible_commands = [
        "Fly to the moon",
        "Teleport to the kitchen",
        "Pick up an object 50 meters away"
    ]
    
    print("Testing Infeasible Commands:")
    print("="*60)
    
    for cmd in infeasible_commands:
        print(f"\nCommand: '{cmd}'")
        try:
            plan = planner.generate_plan(cmd)
            print(f"  Feasible: {plan.feasibility.is_feasible}")
            print(f"  Confidence: {plan.feasibility.confidence:.2%}")
            if plan.feasibility.warnings:
                print("  Warnings:")
                for warning in plan.feasibility.warnings:
                    print(f"    - {warning}")
        except Exception as e:
            print(f"  Error: {e}")
else:
    print("⚠️ Planner not initialized (API key required)")

## Part 9: Retry Logic and Error Handling

LLM APIs can fail or return invalid JSON. Robust systems need retry logic.

In [ ]:
# Demonstrate retry mechanism
if planner:
    # Ambiguous command that might fail
    ambiguous_cmd = "Do something with the thing"
    
    print(f"Command: '{ambiguous_cmd}'")
    print("\nThis may require retries due to ambiguity...\n")
    
    try:
        plan = planner.generate_plan(ambiguous_cmd, max_retries=3)
        print("\n✓ Plan generated successfully")
        print(f"Tasks: {[t.task_type for t in plan.tasks]}")
    except ValueError as e:
        print(f"\n❌ Failed after retries: {e}")
        print("\nLearning: Provide clear, specific commands for better results")
else:
    print("⚠️ Planner not initialized (API key required)")

print("\nRetry Strategy:")
print("1. Exponential backoff: 2^attempt seconds")
print("2. Maximum attempts: 3")
print("3. JSON validation on each attempt")
print("4. Clear error messages for debugging")

## Part 10: Safety Constraints

Robot planning must enforce safety rules.

In [ ]:
def validate_safety_constraints(plan: TaskPlan) -> dict:
    """
    Validate plan against safety constraints.
    
    Returns:
        Dictionary with validation results
    """
    violations = []
    
    # Rule 1: Navigate before manipulate
    manipulate_tasks = [t for t in plan.tasks if t.task_type == 'manipulate']
    for m_task in manipulate_tasks:
        # Check if there's a navigate task in preconditions
        has_nav_precond = any(
            plan.tasks[i-1].task_type == 'navigate' 
            for i in m_task.preconditions 
            if 0 < i <= len(plan.tasks)
        )
        if not has_nav_precond:
            violations.append(f"Task {m_task.task_id}: Manipulation without prior navigation")
    
    # Rule 2: Detect before manipulate
    for m_task in manipulate_tasks:
        has_detect_precond = any(
            plan.tasks[i-1].task_type == 'detect'
            for i in m_task.preconditions
            if 0 < i <= len(plan.tasks)
        )
        if not has_detect_precond:
            violations.append(f"Task {m_task.task_id}: Manipulation without prior detection")
    
    # Rule 3: Workspace bounds
    for task in plan.tasks:
        if 'position' in task.parameters:
            pos = task.parameters['position']
            if len(pos) == 3:
                x, y, z = pos
                if not (-5 <= x <= 5):
                    violations.append(f"Task {task.task_id}: X position {x} out of bounds [-5, 5]")
                if not (-5 <= y <= 5):
                    violations.append(f"Task {task.task_id}: Y position {y} out of bounds [-5, 5]")
                if not (0 <= z <= 3):
                    violations.append(f"Task {task.task_id}: Z position {z} out of bounds [0, 3]")
    
    return {
        'valid': len(violations) == 0,
        'violations': violations
    }

# Test safety validation
if 'complex_plan' in locals():
    safety_result = validate_safety_constraints(complex_plan)
    
    print("Safety Validation Results:")
    print("="*60)
    print(f"Valid: {safety_result['valid']}")
    if safety_result['violations']:
        print("\nViolations:")
        for v in safety_result['violations']:
            print(f"  ⚠️ {v}")
    else:
        print("✓ No safety violations detected")
else:
    print("⚠️ No plan to validate (run previous cells first)")

## Part 11: Cost Analysis

LLM API calls have costs. Let's analyze usage.

In [ ]:
if 'complex_plan' in locals():
    # GPT-4 pricing (as of 2024)
    PRICE_PER_1K_PROMPT = 0.03  # $0.03 per 1K prompt tokens
    PRICE_PER_1K_COMPLETION = 0.06  # $0.06 per 1K completion tokens
    
    metadata = complex_plan.metadata
    
    prompt_cost = (metadata.prompt_tokens / 1000) * PRICE_PER_1K_PROMPT
    completion_cost = (metadata.completion_tokens / 1000) * PRICE_PER_1K_COMPLETION
    total_cost = prompt_cost + completion_cost
    
    print("LLM API Cost Analysis:")
    print("="*60)
    print(f"Model: {metadata.llm_model}")
    print(f"Prompt tokens: {metadata.prompt_tokens:,}")
    print(f"Completion tokens: {metadata.completion_tokens:,}")
    print(f"Total tokens: {metadata.prompt_tokens + metadata.completion_tokens:,}")
    print(f"\nCost breakdown:")
    print(f"  Prompt: ${prompt_cost:.4f}")
    print(f"  Completion: ${completion_cost:.4f}")
    print(f"  Total: ${total_cost:.4f}")
    print(f"\nGeneration time: {metadata.generation_time:.2f}s")
    print(f"Cost per second: ${total_cost/metadata.generation_time:.4f}/s")
    
    # Estimate for 1000 plans
    print(f"\nEstimate for 1,000 similar plans: ${total_cost * 1000:.2f}")
else:
    print("⚠️ No plan metadata available (run previous cells first)")

## Exercise: Design Your Own Task Plan

### Task:
Create a plan for the command: "Clean the living room by picking up all trash and placing it in the bin"

### Requirements:
1. Generate the plan using the LLM planner
2. Visualize the task dependency graph
3. Validate safety constraints
4. Analyze the cost
5. Modify the plan to be more efficient (fewer tasks)

### Questions:
1. How many tasks were generated?
2. What are the dependencies between tasks?
3. Are there any safety violations?
4. How could you improve the prompt to get better plans?

In [ ]:
# YOUR CODE HERE
# Try the exercise above!

if planner:
    exercise_command = "Clean the living room by picking up all trash and placing it in the bin"
    
    # TODO: Generate plan
    # TODO: Visualize dependencies
    # TODO: Validate safety
    # TODO: Analyze cost
    pass
else:
    print("⚠️ Planner not initialized (API key required)")

## Summary

In this notebook, you learned:

✅ **Task Planning Fundamentals**
- Converting natural language to structured plans
- Four task types: navigate, detect, manipulate, inspect
- Task dependencies and preconditions

✅ **Prompt Engineering**
- System prompt design
- Safety constraints
- JSON schema enforcement

✅ **LLM Integration**
- OpenAI API usage
- Retry logic with exponential backoff
- Error handling strategies

✅ **Plan Validation**
- Feasibility assessment
- Safety constraint checking
- Dependency graph visualization

✅ **Cost Analysis**
- Token usage tracking
- Cost per plan calculation
- Optimization strategies

### Next Steps:
- Proceed to `03_object_detection.ipynb` for vision integration
- Experiment with different LLM models (GPT-3.5 vs GPT-4)
- Try local LLMs (Ollama, LLaMA) for cost savings
- Implement custom safety validators for your use case